# Week 5: Searching — Linear vs Binary Search — PHASE 3 "Choosing Between O(n), O(log n), O(1)"

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Understand what searching means and why efficient search matters
2. Implement **linear search** and understand its O(n) time complexity
3. Implement **binary search** and understand its O(log n) time complexity
4. Recognize that binary search **requires sorted data**
5. Trace through binary search step by step
6. Use Python's built-in `bisect` module for binary search
7. Benchmark linear vs binary search and interpret performance plots

## 🎯 Core Mastery Connection

O(n) vs O(log n) — binary search is dramatically faster but requires sorted data. This is a core trade-off: you pay upfront cost (sorting) to gain repeated search speed. This week you will benchmark both search algorithms and see that at n=1,000,000, linear search needs a million comparisons while binary search needs only 20. The data structure choice (sorted vs unsorted) determines which algorithm you can use.

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import bisect
import math
import random
import time

---
## Part 1: What Is Searching?

**Searching** = finding a specific item (the **target**) in a collection of items.

### Real-World Analogies

| Scenario | Search Type |
|---|---|
| Looking for your keys by checking every pocket one by one | **Linear search** |
| Looking up a word in a dictionary by opening to the middle, then narrowing down | **Binary search** |
| Finding a name in an unsorted pile of papers | **Linear search** (no choice) |
| Finding a page in a book using the table of contents | **Binary search** (sorted!) |

### Why Does It Matter?

Imagine searching through 1 million items:
- **Linear search:** up to 1,000,000 comparisons
- **Binary search:** at most **20** comparisons!

That's the power of choosing the right algorithm.

---
## Part 2: Linear Search — Check Every Element

The simplest search: start from the beginning and check each element until you find the target (or reach the end).

**How it works:**
1. Start at index 0
2. Is this element the target? If yes, return the index
3. If not, move to the next element
4. If you reach the end without finding it, return -1 (not found)

**Figure 2.1** — Basic linear search implementation

In [ ]:
def linear_search(lst, target):
    """Search for target in lst. Return index if found, -1 if not."""
    for i in range(len(lst)):
        if lst[i] == target:
            return i
    return -1

# Test it
numbers = [4, 2, 7, 1, 9, 3, 8, 5]
print(f"List: {numbers}")
print(f"Search for 9: index = {linear_search(numbers, 9)}")
print(f"Search for 1: index = {linear_search(numbers, 1)}")
print(f"Search for 6: index = {linear_search(numbers, 6)}  (not found)")

**Figure 2.2** — Linear search with step-by-step tracing

In [ ]:
def linear_search_traced(lst, target):
    """Linear search with detailed step output."""
    print(f"Searching for {target} in {lst}")
    print("-" * 50)
    
    for i in range(len(lst)):
        if lst[i] == target:
            print(f"  Step {i+1}: index {i} -> {lst[i]} == {target}  FOUND!")
            return i
        else:
            print(f"  Step {i+1}: index {i} -> {lst[i]} != {target}  (keep looking)")
    
    print(f"  Reached the end. {target} not found.")
    return -1

data = [12, 45, 7, 23, 56, 89, 34]
linear_search_traced(data, 56)
print()
linear_search_traced(data, 99)

### Linear Search Complexity

| Scenario | Comparisons | Big-O |
|---|---|---|
| Best case (target is first) | 1 | O(1) |
| Worst case (target is last or not found) | n | O(n) |
| Average case | n/2 | O(n) |

**Key point:** Linear search works on **any** list — sorted or unsorted.

---
## Part 3: Binary Search — Divide and Conquer

Binary search is **much faster**, but it has one requirement: the list must be **sorted**.

### The Dictionary Analogy

When you look up "Python" in a dictionary:
1. Open to the **middle** — you see words starting with "M"
2. "Python" comes after "M", so go to the **right half**
3. Open to the middle of the right half — you see "S"
4. "Python" comes before "S", so go to the **left half** of this section
5. Keep halving until you find it!

### How It Works

1. Set `low = 0` and `high = len(lst) - 1`
2. Calculate `mid = (low + high) // 2`
3. If `lst[mid] == target` → found it!
4. If `target < lst[mid]` → search the left half (`high = mid - 1`)
5. If `target > lst[mid]` → search the right half (`low = mid + 1`)
6. Repeat until found or `low > high` (not found)

**Figure 3.1** — Binary search implementation (iterative)

In [ ]:
def binary_search(lst, target):
    """Binary search for target in a SORTED list. Return index or -1."""
    low = 0
    high = len(lst) - 1
    
    while low <= high:
        mid = (low + high) // 2
        
        if lst[mid] == target:
            return mid             # Found!
        elif target < lst[mid]:
            high = mid - 1         # Search left half
        else:
            low = mid + 1          # Search right half
    
    return -1                      # Not found

# Test with a sorted list
sorted_numbers = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(f"Sorted list: {sorted_numbers}")
print(f"Search for 23: index = {binary_search(sorted_numbers, 23)}")
print(f"Search for 2:  index = {binary_search(sorted_numbers, 2)}")
print(f"Search for 50: index = {binary_search(sorted_numbers, 50)}  (not found)")

**Figure 3.2** — Binary search with step-by-step visualization

In [ ]:
def binary_search_traced(lst, target):
    """Binary search with detailed step-by-step output."""
    low = 0
    high = len(lst) - 1
    step = 0
    
    print(f"Searching for {target} in {lst}")
    print("=" * 60)
    
    while low <= high:
        mid = (low + high) // 2
        step += 1
        
        # Visualize the current search range
        markers = ["   "] * len(lst)
        for i in range(low, high + 1):
            markers[i] = " . "
        markers[mid] = " ^ "
        
        print(f"\n  Step {step}: low={low}, high={high}, mid={mid}")
        print(f"  Values:  {lst}")
        print(f"  Range:  {''.join(markers)}")
        
        if lst[mid] == target:
            print(f"  lst[{mid}] = {lst[mid]} == {target} -> FOUND at index {mid}!")
            return mid
        elif target < lst[mid]:
            print(f"  lst[{mid}] = {lst[mid]} > {target} -> search LEFT half")
            high = mid - 1
        else:
            print(f"  lst[{mid}] = {lst[mid]} < {target} -> search RIGHT half")
            low = mid + 1
    
    print(f"\n  low ({low}) > high ({high}) -> {target} NOT FOUND")
    return -1

data = [3, 7, 11, 15, 19, 23, 27, 31, 35, 39]
binary_search_traced(data, 23)

**Figure 3.3** — Binary search: a case where the target is not found

In [ ]:
data = [3, 7, 11, 15, 19, 23, 27, 31, 35, 39]
binary_search_traced(data, 20)

---
## Part 4: Binary Search — Recursive Version

Binary search naturally fits recursion: each step solves a smaller version of the same problem.

**Figure 4.1** — Recursive binary search implementation

In [ ]:
def binary_search_recursive(lst, target, low=0, high=None):
    """Binary search using recursion."""
    if high is None:
        high = len(lst) - 1
    
    if low > high:             # Base case: not found
        return -1
    
    mid = (low + high) // 2
    
    if lst[mid] == target:     # Base case: found
        return mid
    elif target < lst[mid]:    # Recursive case: search left
        return binary_search_recursive(lst, target, low, mid - 1)
    else:                      # Recursive case: search right
        return binary_search_recursive(lst, target, mid + 1, high)

# Test
data = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
print(f"List: {data}")
for target in [7, 1, 19, 10]:
    result = binary_search_recursive(data, target)
    status = f"found at index {result}" if result != -1 else "not found"
    print(f"  Search for {target:2d}: {status}")

---
## Part 5: Big-O Comparison — O(n) vs O(log n)

| List Size (n) | Linear Search (worst) | Binary Search (worst) |
|---:|---:|---:|
| 10 | 10 | 4 |
| 100 | 100 | 7 |
| 1,000 | 1,000 | 10 |
| 1,000,000 | 1,000,000 | 20 |
| 1,000,000,000 | 1,000,000,000 | 30 |

Binary search cuts the problem in half at each step, so it only needs **log₂(n)** steps.

**Figure 5.1** — Counting comparisons for both algorithms

In [ ]:
def linear_search_count(lst, target):
    """Linear search that counts comparisons."""
    comparisons = 0
    for i in range(len(lst)):
        comparisons += 1
        if lst[i] == target:
            return i, comparisons
    return -1, comparisons

def binary_search_count(lst, target):
    """Binary search that counts comparisons."""
    comparisons = 0
    low, high = 0, len(lst) - 1
    
    while low <= high:
        comparisons += 1
        mid = (low + high) // 2
        if lst[mid] == target:
            return mid, comparisons
        elif target < lst[mid]:
            high = mid - 1
        else:
            low = mid + 1
    return -1, comparisons

import math

print(f"{'List Size':>12}  {'Linear (worst)':>16}  {'Binary (worst)':>16}")
print("-" * 50)
for size in [10, 100, 1000, 10_000, 100_000, 1_000_000]:
    data = list(range(size))       # Sorted list
    target = -1                    # Worst case: not found
    _, lin_comp = linear_search_count(data, target)
    _, bin_comp = binary_search_count(data, target)
    print(f"{size:>12,}  {lin_comp:>16,}  {bin_comp:>16,}")

**Figure 5.2** — Visualizing O(n) vs O(log n) growth

In [ ]:
import matplotlib.pyplot as plt
import math

ns = list(range(1, 1001))
linear_ops = ns                                  # O(n)
binary_ops = [math.ceil(math.log2(n + 1)) for n in ns]  # O(log n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Both on same scale
ax1.plot(ns, linear_ops, 'r-', label='Linear O(n)', linewidth=2)
ax1.plot(ns, binary_ops, 'g-', label='Binary O(log n)', linewidth=2)
ax1.set_xlabel('List Size (n)', fontsize=12)
ax1.set_ylabel('Worst-Case Comparisons', fontsize=12)
ax1.set_title('Linear vs Binary Search', fontsize=14)
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Plot 2: Log scale to see both clearly
ax2.plot(ns, linear_ops, 'r-', label='Linear O(n)', linewidth=2)
ax2.plot(ns, binary_ops, 'g-', label='Binary O(log n)', linewidth=2)
ax2.set_xlabel('List Size (n)', fontsize=12)
ax2.set_ylabel('Comparisons (log scale)', fontsize=12)
ax2.set_title('Log Scale View', fontsize=14)
ax2.set_yscale('log')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The red line (linear) grows steadily.")
print("The green line (binary) barely grows at all!")

---
## Part 6: Important Rule — Binary Search Needs Sorted Data!

**Figure 6.1** — What happens when you use binary search on unsorted data

In [ ]:
# DANGER: Binary search on UNSORTED data gives WRONG results!
unsorted = [8, 3, 11, 1, 15, 7, 4, 9]
target = 7

# Linear search (correct)
lin_result = linear_search(unsorted, target)
print(f"Unsorted list: {unsorted}")
print(f"Linear search for {target}: index {lin_result} -> value {unsorted[lin_result]}")

# Binary search (INCORRECT on unsorted data!)
bin_result = binary_search(unsorted, target)
if bin_result == -1:
    print(f"Binary search for {target}: NOT FOUND (WRONG! It's actually there!)")
else:
    print(f"Binary search for {target}: index {bin_result} -> value {unsorted[bin_result]}")

print("\nLesson: ALWAYS sort first if you want to use binary search!")

# Correct approach
sorted_data = sorted(unsorted)
print(f"\nSorted list: {sorted_data}")
print(f"Binary search for {target}: index {binary_search(sorted_data, target)}")

**Figure 6.2** — When to use which search?

| Situation | Best Choice | Why? |
|---|---|---|
| Unsorted data, search once | Linear search | No sorting cost |
| Sorted data | Binary search | O(log n) is much faster |
| Unsorted data, search many times | Sort first, then binary search | Sorting cost O(n log n) pays off over many searches |
| Very small list (< 20 items) | Either works | Difference is negligible |

---
## Part 7: Python's `bisect` Module

Python has a built-in module for binary search operations on sorted lists.

**Figure 7.1** — Using bisect for binary search

In [ ]:
import bisect

sorted_data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(f"Sorted list: {sorted_data}")

# bisect_left: find insertion point (leftmost position)
target = 23
pos = bisect.bisect_left(sorted_data, target)
print(f"\nbisect_left({target}) = {pos}")

# Check if target actually exists at that position
if pos < len(sorted_data) and sorted_data[pos] == target:
    print(f"  -> {target} found at index {pos}")
else:
    print(f"  -> {target} not found")

# Searching for a value that doesn't exist
target = 20
pos = bisect.bisect_left(sorted_data, target)
print(f"\nbisect_left({target}) = {pos}")
if pos < len(sorted_data) and sorted_data[pos] == target:
    print(f"  -> {target} found at index {pos}")
else:
    print(f"  -> {target} not found (would be inserted at index {pos})")

**Figure 7.2** — Using bisect.insort to insert and keep sorted order

In [ ]:
import bisect

# Start with a sorted list
scores = [55, 65, 72, 85, 91]
print(f"Original: {scores}")

# Insert new scores while keeping the list sorted
new_scores = [78, 60, 95, 70]
for score in new_scores:
    bisect.insort(scores, score)
    print(f"Insert {score}: {scores}")

print(f"\nFinal sorted list: {scores}")

**Figure 7.3** — Helper function: clean binary search using bisect

In [ ]:
import bisect

def binary_search_bisect(sorted_lst, target):
    """Binary search using Python's bisect module."""
    i = bisect.bisect_left(sorted_lst, target)
    if i < len(sorted_lst) and sorted_lst[i] == target:
        return i
    return -1

# Test
data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
for target in [30, 55, 100, 5]:
    result = binary_search_bisect(data, target)
    status = f"found at index {result}" if result != -1 else "not found"
    print(f"Search for {target:3d}: {status}")

---
## Part 8: Benchmarking — Linear vs Binary Search

> **🔮 Predict first, then measure. Does reality match your prediction?** Before running: predict how many times faster binary search will be at n=1,000,000. Write your prediction down, then check.

**Figure 8.1** — Timing both search algorithms on sorted lists of increasing size

In [ ]:
import time
import random

def benchmark_search(search_func, data, target, repeats=100):
    """Time a search function, averaging over multiple runs."""
    start = time.perf_counter()
    for _ in range(repeats):
        search_func(data, target)
    elapsed = (time.perf_counter() - start) / repeats
    return elapsed

sizes = [100, 500, 1000, 5000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
linear_times = []
binary_times = []

print(f"{'Size':>12}  {'Linear (sec)':>14}  {'Binary (sec)':>14}  {'Ratio':>10}")
print("-" * 56)

for size in sizes:
    data = list(range(size))   # Sorted list
    target = -1                # Worst case: not in list
    
    lt = benchmark_search(linear_search, data, target, repeats=10)
    bt = benchmark_search(binary_search, data, target, repeats=10)
    
    linear_times.append(lt)
    binary_times.append(bt)
    
    ratio = lt / bt if bt > 0 else float('inf')
    print(f"{size:>12,}  {lt:>14.8f}  {bt:>14.8f}  {ratio:>10.0f}x")

**Figure 8.2** — Plotting the benchmark results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Both on normal scale
ax1.plot(sizes, linear_times, 'ro-', label='Linear Search', linewidth=2, markersize=6)
ax1.plot(sizes, binary_times, 'gs-', label='Binary Search', linewidth=2, markersize=6)
ax1.set_xlabel('List Size', fontsize=12)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Search Time vs List Size', fontsize=14)
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Plot 2: Log-log scale
ax2.loglog(sizes, linear_times, 'ro-', label='Linear Search', linewidth=2, markersize=6)
ax2.loglog(sizes, binary_times, 'gs-', label='Binary Search', linewidth=2, markersize=6)
ax2.set_xlabel('List Size (log scale)', fontsize=12)
ax2.set_ylabel('Time (seconds, log scale)', fontsize=12)
ax2.set_title('Log-Log Scale View', fontsize=14)
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Observations:")
print("- Linear search time grows proportionally with list size")
print("- Binary search time barely changes even for huge lists")
print("- The gap between them widens dramatically as size increases")

---
## Part 9: Common Errors with Searching

**Figure 9.1** — Error: Off-by-one in binary search boundaries

In [ ]:
# COMMON BUG: Using high = mid instead of high = mid - 1
def binary_search_buggy(lst, target):
    low, high = 0, len(lst) - 1
    iterations = 0
    while low <= high:
        iterations += 1
        if iterations > 100:  # Safety limit
            print("  INFINITE LOOP detected! Breaking out.")
            return -1
        mid = (low + high) // 2
        if lst[mid] == target:
            return mid
        elif target < lst[mid]:
            high = mid     # BUG! Should be mid - 1
        else:
            low = mid      # BUG! Should be mid + 1
    return -1

data = [1, 3, 5, 7, 9]
print("Buggy binary search (may infinite loop):")
result = binary_search_buggy(data, 2)  # Not in list
print(f"Result: {result}")

print("\nFix: Always use mid - 1 and mid + 1 to shrink the range!")

**Figure 9.2** — Error: Using binary search on unsorted data

In [ ]:
# Demonstrate the problem clearly
unsorted = [50, 20, 80, 10, 60, 30, 90, 40, 70]
target = 60

# Linear search: always correct
lin_idx = linear_search(unsorted, target)
print(f"List (unsorted): {unsorted}")
print(f"Looking for: {target}")
print(f"Linear search result: index {lin_idx} (correct)")

# Binary search: unreliable on unsorted data
bin_idx = binary_search(unsorted, target)
print(f"Binary search result: index {bin_idx} (UNRELIABLE!)")

print("\nAlways check: Is my data sorted before using binary search?")

try:
    # A safe wrapper
    def safe_binary_search(lst, target):
        if lst != sorted(lst):
            raise ValueError("List must be sorted for binary search!")
        return binary_search(lst, target)
    
    safe_binary_search(unsorted, target)
except ValueError as e:
    print(f"\nSafe wrapper caught it: {e}")

**Figure 9.3** — Error: Integer overflow in midpoint calculation (important in other languages)

In [ ]:
# In languages like C/Java, (low + high) can overflow!
# Python handles big integers natively, but it's good to know.

# Potentially problematic in other languages:
low = 2_000_000_000
high = 2_000_000_000
mid_bad = (low + high) // 2     # low + high = 4 billion (overflow in 32-bit!)
mid_good = low + (high - low) // 2  # Safe: never exceeds high

print(f"Method 1 (low + high) // 2:       {mid_bad}")
print(f"Method 2 low + (high - low) // 2: {mid_good}")
print(f"Same result in Python: {mid_bad == mid_good}")
print("\nIn Python this doesn't matter (arbitrary precision integers),")
print("but Method 2 is the safe habit for all languages.")

---
## 🌉 Bridge to Next Week

This week we explored **searching** — how to find elements efficiently.

Key takeaways:
- **Linear search** checks every element: simple but slow O(n)
- **Binary search** halves the search space: fast O(log n) but needs sorted data
- Python's `bisect` module provides efficient binary search operations
- Choosing the right algorithm can mean the difference between 1 million operations and 20!

**Next week**, we'll learn about **Sorting** — the algorithms that put data in order. We noticed that binary search needs sorted data... but how do we sort efficiently? We'll explore **Insertion Sort**, **Merge Sort**, and **Quick Sort**, compare their performance, and see how Python's built-in `sorted()` works under the hood.

---
## 🎢 Exercises

Complete each exercise in the code cell below its description. Make sure to **run** each cell before submitting.

> **🔮 Predict first, then measure. Does reality match your prediction?** For every exercise that involves timing or complexity analysis, make your prediction BEFORE running the code.

### Easy

**EX1 — Linear Search with Count**

Write a function `search_count(lst, target)` that returns a tuple `(index, comparisons)`. If the target is not found, return `(-1, comparisons)`.

**Expected Output:**
```
search_count([4, 8, 2, 6, 1], 6) = (3, 4)
search_count([4, 8, 2, 6, 1], 9) = (-1, 5)
search_count([4, 8, 2, 6, 1], 4) = (0, 1)
```

<details><summary>💡 Hint</summary>
Use a counter variable that increments by 1 each time you compare an element to the target.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 — Find All Occurrences**

Write a function `find_all(lst, target)` that returns a list of **all** indices where target appears.

**Expected Output:**
```
find_all([1, 3, 5, 3, 7, 3], 3) = [1, 3, 5]
find_all([1, 2, 3, 4, 5], 6) = []
find_all([7, 7, 7], 7) = [0, 1, 2]
```

<details><summary>💡 Hint</summary>
Create an empty result list. Loop through the list and append the index whenever you find the target.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 — Search in a List of Strings**

Write a function `search_name(names, target)` that searches for a name (case-insensitive) in a list and returns the index, or -1 if not found.

**Expected Output:**
```
search_name(["Alice", "Bob", "Charlie"], "bob") = 1
search_name(["Alice", "Bob", "Charlie"], "BOB") = 1
search_name(["Alice", "Bob", "Charlie"], "Dave") = -1
```

<details><summary>💡 Hint</summary>
Convert both the list element and the target to lowercase using `.lower()` before comparing.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 — Binary Search — Find the Target**

Write a function `my_binary_search(sorted_lst, target)` from scratch (no bisect). Return the index if found, -1 if not.

**Expected Output:**
```
my_binary_search([1, 3, 5, 7, 9, 11, 13], 7) = 3
my_binary_search([1, 3, 5, 7, 9, 11, 13], 1) = 0
my_binary_search([1, 3, 5, 7, 9, 11, 13], 8) = -1
```

<details><summary>💡 Hint</summary>
Follow the pattern: set low and high, loop while low <= high, compute mid, compare and adjust.
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium

**EX5 — Find First and Last Position**

Given a sorted list with duplicates, write `find_range(sorted_lst, target)` that returns a tuple `(first_index, last_index)`. Return `(-1, -1)` if not found.

**Expected Output:**
```
find_range([1, 2, 3, 3, 3, 4, 5], 3) = (2, 4)
find_range([1, 1, 1, 1], 1) = (0, 3)
find_range([1, 2, 3, 4, 5], 6) = (-1, -1)
```

<details><summary>💡 Hint</summary>
Use bisect_left to find the first occurrence and bisect_right - 1 to find the last. Or do two binary searches: one to find the leftmost, one to find the rightmost.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 — Search in a Sorted Matrix**

Given a 2D list where each row is sorted, write `search_matrix(matrix, target)` that returns `(row, col)` or `(-1, -1)`.

**Expected Output:**
```
matrix = [
    [1,  3,  5,  7],
    [10, 11, 16, 20],
    [23, 30, 34, 50]
]
search_matrix(matrix, 16) = (1, 2)
search_matrix(matrix, 25) = (-1, -1)
```

<details><summary>💡 Hint</summary>
For each row, check if the target could be in that row (between first and last element). If so, use binary search on that row.
</details>

In [ ]:
# ✏️ [EX6] Your code here


**EX7 — Find Square Root (Integer)**

Use binary search to find the integer square root of a number: the largest integer `x` where `x*x <= n`.

Write `int_sqrt(n)` using binary search.

**Expected Output:**
```
int_sqrt(16) = 4
int_sqrt(20) = 4
int_sqrt(100) = 10
int_sqrt(1) = 1
```

<details><summary>💡 Hint</summary>
Binary search between 0 and n. If mid*mid <= n, record mid as a candidate and search right. If mid*mid > n, search left.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 — Count Comparisons Experiment**

For list sizes `[100, 1000, 10000, 100000]`, create sorted lists and search for a value NOT in the list (worst case). Print the number of comparisons for both linear and binary search.

**Expected Output (format):**
```
Size: 100     Linear: 100     Binary: 7
Size: 1000    Linear: 1000    Binary: 10
...
```

<details><summary>💡 Hint</summary>
Use the `linear_search_count` and `binary_search_count` functions from Part 5, or write your own versions that count comparisons.
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 — Recursive Binary Search**

Write `rec_binary_search(sorted_lst, target)` using recursion (no while loop).

**Expected Output:**
```
rec_binary_search([2, 4, 6, 8, 10, 12], 8) = 3
rec_binary_search([2, 4, 6, 8, 10, 12], 5) = -1
rec_binary_search([2, 4, 6, 8, 10, 12], 2) = 0
```

<details><summary>💡 Hint</summary>
Define a helper function with low and high parameters. Base case: low > high returns -1. Otherwise compute mid and recurse on the appropriate half.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 — Find Closest Value**

Write `find_closest(sorted_lst, target)` that returns the value in the sorted list closest to the target.

**Expected Output:**
```
find_closest([1, 4, 6, 8, 12], 5) = 4 or 6
find_closest([1, 4, 6, 8, 12], 7) = 6 or 8
find_closest([1, 4, 6, 8, 12], 12) = 12
```

<details><summary>💡 Hint</summary>
Use bisect_left to find where target would be inserted. Then compare the element at that position and the one before it to see which is closer.
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge

**EX11 — Search in Rotated Sorted Array**

A sorted array has been "rotated" (e.g., `[4, 5, 6, 7, 0, 1, 2]` was `[0, 1, 2, 4, 5, 6, 7]`). Write `search_rotated(lst, target)` that finds the target in O(log n).

**Expected Output:**
```
search_rotated([4, 5, 6, 7, 0, 1, 2], 0) = 4
search_rotated([4, 5, 6, 7, 0, 1, 2], 5) = 1
search_rotated([4, 5, 6, 7, 0, 1, 2], 3) = -1
```

<details><summary>💡 Hint</summary>
At each step, one half of the array is always sorted. Check which half is sorted and whether the target lies in that sorted half. If yes, search there; otherwise, search the other half.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 — Peak Finder**

Write `find_peak(lst)` that finds a "peak" element (an element greater than its neighbors) in O(log n) using binary search.

**Expected Output (any valid peak):**
```
find_peak([1, 3, 5, 4, 2]) = 2  (value 5 is a peak)
find_peak([1, 2, 3, 4, 5]) = 4  (value 5 at end is a peak)
find_peak([5, 4, 3, 2, 1]) = 0  (value 5 at start is a peak)
```

<details><summary>💡 Hint</summary>
Compare mid with its neighbors. If mid is greater than both, it's a peak. If the left neighbor is greater, a peak exists in the left half. If the right neighbor is greater, a peak exists in the right half.
</details>

In [ ]:
# ✏️ [EX12] Your code here


---
## 📮 Submission

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 1: Fill in your info below, then run this cell
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STUDENT_ID    = ""     # e.g. "2024001234"
STUDENT_NAME  = ""     # e.g. "Ahmet Y\u0131lmaz"
STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"
CLASS_CODE    = ""     # code given in class
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Don't change anything below this line
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re as _re
_errors = []
if not _re.match(r"^\d{6,12}$", STUDENT_ID):
    _errors.append("\u274c Student ID must be 6-12 digits")
if len(STUDENT_NAME.strip().split()) < 2:
    _errors.append("\u274c Enter first and last name")
if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:
    _errors.append("\u274c Use your @istun.edu.tr email")
if len(CLASS_CODE.strip()) < 4:
    _errors.append("\u274c Invalid class code")
if _errors:
    for _e in _errors:
        print(_e)
    print("\n\u26a0\ufe0f  Fix the errors above and run this cell again.")
else:
    print(f"\u2705 Info OK \u2014 {STUDENT_NAME} ({STUDENT_ID})")
    print(f"   {STUDENT_EMAIL}")
    print(f"\n\ud83d\udc49 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 2: Run this cell to submit
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, re, os, urllib.request
WEEK = "Week_05"
URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"
try:
    _sid = STUDENT_ID.strip()
    _sname = STUDENT_NAME.strip()
    _semail = STUDENT_EMAIL.strip().lower()
    _scode = CLASS_CODE.strip().upper()
except NameError:
    raise SystemExit("\u274c Run the cell above first to set your info!")
if not _sid or not _sname or not _semail or not _scode:
    raise SystemExit("\u274c Run the cell above first \u2014 some fields are empty.")
_answers = {}
try:
    _ipy = get_ipython()
    _hist = _ipy.history_manager.get_range(output=False)
    for _sess, _line, _src in _hist:
        _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
        if _m:
            _ex_id = "ex" + _m.group(1)
            _lines = _src.split("\n")
            _clean = "\n".join(_lines[1:]).strip()
            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
except Exception:
    pass
if not _answers:
    try:
        for _src in In:
            if not _src: continue
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
    except NameError:
        pass
if not _answers:
    _nb_path = None
    try:
        _nb_path = __vsc_ipynb_file__
    except NameError:
        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]
        if len(_candidates) == 1: _nb_path = _candidates[0]
    if _nb_path and os.path.exists(str(_nb_path)):
        with open(str(_nb_path), "r", encoding="utf-8") as _f:
            _nb = json.load(_f)
        for _cell in _nb["cells"]:
            if _cell["cell_type"] != "code": continue
            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
print(f"\ud83d\udcdd Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")
if not _answers:
    print("\n\u26a0\ufe0f  No exercise answers found!")
    print("Make sure you RAN all exercise cells before submitting.")
    raise SystemExit()
_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "dsa-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")
print("\ud83d\udce1 Submitting...")
try:
    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
    _resp = urllib.request.urlopen(_req, timeout=30)
    _result = json.loads(_resp.read().decode())
    if _result.get("success"):
        print(f"\n\u2705 {_result['message']}")
        print("\ud83d\udce7 Check your email for confirmation.")
    else:
        print(f"\n\u274c {_result.get('message', 'Submission failed')}")
except Exception as _e:
    try:
        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
        urllib.request.urlopen(_req, timeout=10)
    except: pass
    print(f"\n\u26a0\ufe0f  Request sent \u2014 check your email for confirmation.")
    print(f"(If no email arrives, try again or contact your instructor)")